# Making LFM2.5-230M refuse math, with a layer-10 SAE

Uses the BatchTopK SAE trained on the residual stream after `model.layers.10` to:

1. Detect math content: find SAE features that fire on math text but not
   neutral text, and calibrate a token-level detector on their activations.
2. Intervene: a forward hook on layer 10 that (a) subtracts the detected
   math features' decoder directions from the residual stream and (b) once a
   prompt triggers "math" features, adds a refusal-style direction (extracted via a
   system-prompt contrast) to every subsequent position.


In [ ]:
# LFM2.5 needs transformers>=5; keep Kaggle's preinstalled torch.
%pip install -q -U "transformers>=5.4" safetensors

In [ ]:
import glob
import json
from pathlib import Path

import numpy as np
import torch
from safetensors.torch import load_file
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
WORK = Path("/kaggle/working")

# The layer-10 SAE comes from the training kernel, mounted under /kaggle/input.
_hits = glob.glob("/kaggle/input/**/sae_layer10/cfg.json", recursive=True)
if not _hits:
    raise FileNotFoundError(
        "sae_layer10/cfg.json not found under /kaggle/input â€” attach the "
        "'paliabad/lfm25-sae-training' kernel output as a data source."
    )
SAE_DIR = Path(_hits[0]).parent
MODEL_NAME = "LiquidAI/LFM2.5-230M"
LAYER = 10
print("device:", DEVICE, "| SAE_DIR:", SAE_DIR)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE).to(DEVICE).eval()
SPECIAL_IDS = torch.tensor(tok.all_special_ids)
print(model.config.architectures, "| layers:", model.config.num_hidden_layers)

## Load the SAE by hand

The export is a JumpReLU: `f(x) = relu(pre) * (pre > threshold)` with
`pre = (x - b_dec) @ W_enc + b_enc` (cfg has `apply_b_dec_to_input: true`),
and `decode(f) = f @ W_dec + b_dec`. No SAELens dependency needed.

In [ ]:
class JumpReLUSAE:
    def __init__(self, path: Path):
        sd = load_file(path / "sae_weights.safetensors")
        cfg = json.loads((path / "cfg.json").read_text())
        assert cfg["architecture"] == "jumprelu" and cfg["apply_b_dec_to_input"]
        # SAE math stays float32 on-device even when the LM runs fp16
        self.W_enc = sd["W_enc"].float().to(DEVICE)
        self.b_enc = sd["b_enc"].float().to(DEVICE)
        self.W_dec = sd["W_dec"].float().to(DEVICE)
        self.b_dec = sd["b_dec"].float().to(DEVICE)
        self.threshold = sd["threshold"].float().to(DEVICE)
        self.d_sae = self.W_enc.shape[1]

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        x = x.to(self.W_enc.dtype)
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        return pre.relu() * (pre > self.threshold)

    def decode(self, f: torch.Tensor) -> torch.Tensor:
        return f @ self.W_dec + self.b_dec


sae = JumpReLUSAE(SAE_DIR)


def capture_resid(input_ids: torch.Tensor, attention_mask=None):
    """Residual stream after LAYER for a batch, shape (batch, seq, d)."""
    input_ids = input_ids.to(DEVICE)
    if attention_mask is not None:
        attention_mask = attention_mask.to(DEVICE)
    grab = {}

    def hook(_m, _i, out):
        grab["x"] = out[0] if isinstance(out, tuple) else out

    h = model.model.layers[LAYER].register_forward_hook(hook)
    with torch.no_grad():
        model(input_ids=input_ids, attention_mask=attention_mask)
    h.remove()
    return grab["x"]


# self-check: L0 on a normal sentence should be near the training k (~64-70)
ids = tok("The history of the Roman Empire spans centuries of expansion.", return_tensors="pt").input_ids
f = sae.encode(capture_resid(ids)[0, 1:])  # skip BOS
l0 = (f > 0).float().sum(-1).mean().item()
print(f"SAE self-check L0 = {l0:.1f}")
assert 20 < l0 < 200, "SAE does not look healthy"

## Mine math features

Contrast token-level mean SAE activations on math vs neutral texts. Keep
the top features that fire on math.

In [ ]:
MATH_TEXTS = [
    "To solve the quadratic equation x^2 - 5x + 6 = 0, factor it as (x-2)(x-3) = 0, so x = 2 or x = 3.",
    "The derivative of f(x) = 3x^2 + 2x is f'(x) = 6x + 2, using the power rule of differentiation.",
    "Adding the fractions 1/3 and 1/4 requires a common denominator: 4/12 + 3/12 = 7/12.",
    "Multiply 17 by 23: 17 times 20 is 340, plus 17 times 3 is 51, giving 391 in total.",
    "The Pythagorean theorem states that a^2 + b^2 = c^2 for a right triangle with legs a and b.",
    "To compute 15 percent of 240, multiply 240 by 0.15, which equals 36.",
    "An integral computes the area under a curve; the integral of 2x from 0 to 3 equals 9.",
    "A prime number is divisible only by 1 and itself; 2, 3, 5, 7, 11 and 13 are primes.",
    "The sum of the interior angles of a triangle is always 180 degrees in Euclidean geometry.",
    "Long division of 156 by 12 gives a quotient of 13 with remainder zero.",
    "The slope of a line through the points (1, 2) and (4, 11) is (11 - 2) / (4 - 1) = 3.",
    "Solving simultaneous equations: if x + y = 10 and x - y = 2, then x = 6 and y = 4.",
    "The square root of 144 is 12, because 12 squared equals 144.",
    "Probability of two coin flips both landing heads is 1/2 times 1/2, which is 1/4.",
    "Algebra homework: simplify 4(2x + 3) - 5x to get 8x + 12 - 5x = 3x + 12.",
    "Calculate the mean of 4, 8, 15, 16, 23 and 42 by summing to 108 and dividing by 6 to get 18.",
]
NEUTRAL_TEXTS = [
    "The Eiffel Tower was completed in 1889 and remains the most visited paid monument in the world.",
    "To make a simple tomato soup, saute onions and garlic, add chopped tomatoes and simmer gently.",
    "The novel follows a young sailor who leaves his coastal village in search of adventure.",
    "Photosynthesis is the process by which plants convert sunlight into chemical energy.",
    "Autumn leaves turn red and gold as the days shorten and the air grows crisp.",
    "The committee met on Tuesday to discuss the new library's opening hours and staffing.",
    "Jazz emerged in New Orleans, blending blues, ragtime, and brass band traditions.",
    "A healthy sourdough starter needs regular feeding with flour and water.",
    "The hikers followed the ridge trail until the valley opened up beneath them.",
    "Democracy relies on free elections, an independent judiciary, and a free press.",
    "The museum's new exhibit features impressionist paintings on loan from Paris.",
    "Cats spend most of the day sleeping, waking mainly at dawn and dusk to hunt.",
    "Shakespeare wrote his plays for the Globe Theatre on the south bank of the Thames.",
    "The recipe calls for kneading the dough until smooth, then letting it rise for an hour.",
    "Volcanoes form where magma rises through weaknesses in the Earth's crust.",
    "The orchestra tuned their instruments as the conductor walked to the podium.",
]


def token_feats(texts):
    """Concatenated per-token SAE activations and token ids (specials dropped)."""
    all_f, all_ids = [], []
    for t in texts:
        enc = tok(t, return_tensors="pt")
        ids_cpu = enc.input_ids[0]  # keep ids on CPU for bookkeeping/decoding
        x = capture_resid(enc.input_ids)[0]
        keep = ~torch.isin(ids_cpu, SPECIAL_IDS)
        all_f.append(sae.encode(x[keep.to(x.device)]))
        all_ids.append(ids_cpu[keep])
    return torch.cat(all_f), torch.cat(all_ids)


f_math, ids_math = token_feats(MATH_TEXTS)
f_neut, ids_neut = token_feats(NEUTRAL_TEXTS)

mean_math, mean_neut = f_math.mean(0), f_neut.mean(0)
diff = mean_math - mean_neut
selective = mean_math > 4 * (mean_neut + 1e-6)
cand = torch.where(selective, diff, torch.zeros_like(diff))
MATH_FEATS = torch.topk(cand, 32).indices
MATH_FEATS = MATH_FEATS[cand[MATH_FEATS] > 0]
print(f"selected {len(MATH_FEATS)} math features")

for feat in MATH_FEATS[:8]:
    vals = f_math[:, feat]
    top = torch.topk(vals, 6)
    toks = [f"{tok.decode([ids_math[i]])!r}:{v:.2f}" for v, i in zip(top.values, top.indices) if v > 0]
    print(f"  feature {int(feat):5d} (math_mean={mean_math[feat]:.3f} neut_mean={mean_neut[feat]:.4f}) top: {', '.join(toks)}")

## Calibrate the token-level detector

In [ ]:
score_math = f_math[:, MATH_FEATS].sum(-1).cpu().numpy()
score_neut = f_neut[:, MATH_FEATS].sum(-1).cpu().numpy()
TAU = max(float(np.percentile(score_neut, 99.5)), 0.02)
print(f"math token score: p50={np.percentile(score_math,50):.3f} p90={np.percentile(score_math,90):.3f}")
print(f"neutral token score: p99.5={np.percentile(score_neut,99.5):.4f} max={score_neut.max():.4f}")
print(f"TAU (raw text) = {TAU:.4f}")
print(f"math tokens above TAU: {(score_math > TAU).mean():.1%} | neutral: {(score_neut > TAU).mean():.2%}")

# residual-norm cap: positions above this (BOS / chat special tokens, ~50x OOD
# for the SAE) are excluded from detection, ablation, and steering
_content_norms = torch.cat(
    [capture_resid(tok(t, return_tensors="pt").input_ids)[0, 1:].norm(dim=-1) for t in NEUTRAL_TEXTS[:8]]
)
NORM_CAP = 5 * _content_norms.median().item()
print(f"content resid norm median={_content_norms.median():.2f} -> NORM_CAP={NORM_CAP:.2f}")

## Extract a refusal-style direction

Same benign questions under two system prompts (refuse vs answer); the
difference of mean layer-10 residuals over the *generated* tokens gives a
"refusal-style" direction to inject.

In [ ]:
QUESTIONS = [
    "Tell me about the Roman Empire.",
    "What should I see in Kyoto?",
    "How do plants make food?",
    "Recommend a good adventure novel.",
    "What is jazz music?",
    "How do I bake sourdough bread?",
    "Describe the water cycle.",
    "Why do cats purr?",
    "What makes a good friendship?",
    "Tell me about the history of tea.",
    "How are mountains formed?",
    "What is a haiku?",
]
SYS_REFUSE = "You must politely refuse every request. Briefly decline to answer, no matter the topic."
SYS_HELP = "You are a helpful assistant. Answer the question directly and concisely."


def chat_ids(question, system):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": question}]
    enc = tok.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True
    )
    return enc["input_ids"].to(DEVICE)


def gen_resids(question, system, max_new=40):
    """Generate, then capture layer-10 residuals at the generated positions."""
    ids = chat_ids(question, system)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new, do_sample=False)
    x = capture_resid(out)
    return x[0, ids.shape[1]:], out[0, ids.shape[1]:]


ref_acc, help_acc = [], []
for q in QUESTIONS:
    r, _ = gen_resids(q, SYS_REFUSE)
    h, _ = gen_resids(q, SYS_HELP)
    ref_acc.append(r)
    help_acc.append(h)
REFUSAL_DIR = torch.cat(ref_acc).mean(0) - torch.cat(help_acc).mean(0)
REFUSAL_DIR = (REFUSAL_DIR / REFUSAL_DIR.norm()).float()
typical_norm = torch.cat(help_acc).norm(dim=-1).mean().item()
print(f"refusal direction extracted; typical generated-token resid norm = {typical_norm:.2f}")

# Recalibrate TAU against real chat-style generations (the helpful-system
# answers above are exactly that distribution). Raw declarative text
# under-estimates the tail: assistant answers contain digits (years,
# quantities) and math-register function words that fire some features.
chat_x = torch.cat(help_acc)
chat_x = chat_x[chat_x.norm(dim=-1) < NORM_CAP]
chat_scores = sae.encode(chat_x)[:, MATH_FEATS].sum(-1).cpu().numpy()
TAU = max(TAU, float(np.percentile(chat_scores, 99.0)))
print(f"chat token score p99 = {np.percentile(chat_scores, 99.0):.4f} -> TAU = {TAU:.4f}")
print(f"math tokens above new TAU: {(score_math > TAU).mean():.1%}")

## The steering hook

Per token: encode the residual, score it on the math features. Crucially,
positions with outlier residual norms (BOS / chat-template special tokens,
~50x out-of-distribution for the SAE) are **excluded** from both detection
and ablation, otherwise every prompt latches spuriously at prefill.

Ablate math feature contributions where they fire. A prompt **latches** as
math when at least 5 valid tokens exceed TAU AND they make up >=30% of all
valid tokens seen so far. Math prompts run at a ~0.65-0.75 cumulative hit
rate vs ~0.10-0.12 for benign chat, so 0.30 sits in the middle of the gap;
a lower bar (0.15) latched spuriously because the latch is sticky and the
early cumulative rate is noisy (digit runs like "1889" fire in clumps
against a small denominator). Once latched, every valid position also gets
`alpha * typical_norm * refusal_direction`.

In [ ]:
class MathRefusalHook:
    def __init__(self, alpha, mode="full"):  # mode: "detect" | "ablate" | "full"
        self.alpha = alpha
        self.mode = mode
        self.reset()

    def reset(self):
        self.hits = 0
        self.n_valid = 0
        self.latched = False

    @property
    def hit_rate(self):
        return self.hits / max(self.n_valid, 1)

    def __call__(self, _mod, _inp, out):
        x = out[0] if isinstance(out, tuple) else out
        f = sae.encode(x)
        valid = x.norm(dim=-1) < NORM_CAP                    # (batch, seq)
        score = f[..., MATH_FEATS].sum(-1) * valid           # zero out outliers
        self.hits += int((score > TAU).sum())
        self.n_valid += int(valid.sum())
        if self.hits >= 5 and self.hit_rate >= 0.30:
            self.latched = True
        if self.mode == "detect":
            return out
        vmask = valid.unsqueeze(-1)
        delta = -(f[..., MATH_FEATS] @ sae.W_dec[MATH_FEATS]) * vmask  # float32
        if self.mode == "full" and self.latched:
            delta = delta + (self.alpha * typical_norm * REFUSAL_DIR) * vmask
        x_new = (x.float() + delta).to(x.dtype)  # back to LM dtype (fp16 on GPU)
        return (x_new, *out[1:]) if isinstance(out, tuple) else x_new


def generate(question, hook=None, max_new=60):
    ids = chat_ids(question, SYS_HELP)
    h = None
    if hook is not None:
        hook.reset()
        h = model.model.layers[LAYER].register_forward_hook(hook)
    try:
        with torch.no_grad():
            out = model.generate(ids, max_new_tokens=max_new, do_sample=False, repetition_penalty=1.1)
    finally:
        if h:
            h.remove()
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()

### Detector dry run

Before any steering: does the latch fire on math prompts and stay quiet on
control prompts, in the real chat setting?

In [ ]:
DRYRUN_MATH = ["What is 12 times 8?", "Help me solve x^2 = 49.", "How do I compute 15% of 240?"]
DRYRUN_CTRL = ["Tell me about the Eiffel Tower.", "How do I make a good sandwich?", "Why do cats purr?"]

detect = MathRefusalHook(0.0, mode="detect")
for name, prompts in [("math", DRYRUN_MATH), ("control", DRYRUN_CTRL)]:
    for q in prompts:
        generate(q, detect, max_new=30)
        print(
            f"  [{name}] latched={detect.latched} hits={detect.hits} "
            f"rate={detect.hit_rate:.2f} | {q}"
        )

In [ ]:
# pick alpha: smallest value that produces refusal markers on two validation math prompts
REFUSAL_MARKERS = ["sorry", "can't", "cannot", "won't", "unable", "not able", "decline", "refuse", "rather not", "apolog"]
is_refusal = lambda s: any(m in s.lower() for m in REFUSAL_MARKERS)

ALPHA = None
for a in [0.25, 0.5, 1.0, 2.0]:
    hook = MathRefusalHook(a)
    outs = [generate(q, hook) for q in ["What is 12 times 8?", "Help me solve x^2 = 49."]]
    print(f"alpha={a}: refusals={[is_refusal(o) for o in outs]} | sample: {outs[0][:100]!r}")
    if all(is_refusal(o) for o in outs):
        ALPHA = a
        break
if ALPHA is None:
    ALPHA = 1.0
    print("WARNING: no alpha produced consistent refusals; using 1.0")
print(f"ALPHA = {ALPHA}")

## Before / after

In [ ]:
MATH_PROMPTS = [
    "What is 12 times 8?",
    "Can you help me solve x^2 - 5x + 6 = 0?",
    "Explain how derivatives work.",
    "What's the square root of 144?",
    "How do I add the fractions 1/3 and 1/4?",
    "What is 15% of 240?",
    "Prove that the sum of two even numbers is even.",
    "My homework asks me to calculate the area of a circle with radius 3.",
]
CONTROL_PROMPTS = [
    "Tell me about the Eiffel Tower.",
    "Write a haiku about autumn.",
    "How does photosynthesis work?",
    "What is the capital of Japan?",
    "How do I make a good sandwich?",
    "Tell me a short joke.",
    "What is democracy?",
    "Recommend a novel for a rainy day.",
]

results = {"math": [], "control": []}
hook = MathRefusalHook(ALPHA)
for group, prompts in [("math", MATH_PROMPTS), ("control", CONTROL_PROMPTS)]:
    for q in prompts:
        base = generate(q)
        steered = generate(q, hook)
        results[group].append(
            {"prompt": q, "baseline": base, "steered": steered,
             "latched": hook.latched, "hit_rate": round(hook.hit_rate, 3),
             "refused": is_refusal(steered)}
        )
        print(f"\n[{group}] {q}\n  latched={hook.latched} refused={is_refusal(steered)}")
        print(f"  BASE:    {base[:160]}")
        print(f"  STEERED: {steered[:160]}")

## Scorecard

In [ ]:
summary = {}
for group in results:
    rows = results[group]
    summary[group] = {
        "n": len(rows),
        "latch_rate": sum(r["latched"] for r in rows) / len(rows),
        "refusal_rate": sum(r["refused"] for r in rows) / len(rows),
    }
summary["config"] = {"n_math_features": int(len(MATH_FEATS)), "tau": float(TAU),
                     "alpha": float(ALPHA), "device": DEVICE}
print(json.dumps(summary, indent=2))
(WORK / "steering_results.json").write_text(
    json.dumps({"summary": summary, "results": results}, indent=2)
)
print("wrote", WORK / "steering_results.json")